In [14]:
def initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose):
    num_evals_init = num_evals
    init_range = hi - lo
    if is_pos:
        mse, z, ghat = eval_fn(hi)
        while mse < epsilon_squared and num_evals > 0:
            hi += init_range
            mse, z, ghat = eval_fn(hi)
            num_evals -= 1
        bound_dict = {'lo':mse_z_ghat_0, 'hi':(mse, z, ghat)}
    else:
        mse, z, ghat = eval_fn(lo)
        while mse > epsilon_squared and num_evals > 0: # remember epsilon_squared will be negative in this case
            lo -= init_range
            mse, z, ghat = eval_fn(lo)
            num_evals -= 1
        bound_dict = {'lo':(mse, z, ghat), 'hi':mse_z_ghat_0}
    if num_evals == 0:
        raise ValueError('Exceeded number of allowable evaluations during initialization of search bounds.')
    elif verbose:
        print(f'Initial bounds: ({lo}, {hi})')
        print(f'{num_evals}/{num_evals_init} evaluations remaining after initialization.')
    return lo, hi, bound_dict, num_evals

def bisection_search(lo, hi , eval_fn, num_evals, epsilon_squared, mse_z_ghat_0, verbose, tol=0):
    is_pos = hi > 0
    lo, hi, bound_dict, num_evals = initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose)
    while lo < hi - tol and num_evals > 0:
        mid = (lo + hi) / 2
        mse, z, ghat = eval_fn(mid)
        if verbose:
            print(f'alpha:{mid}, mse:{mse}, lo:{lo}, hi:{hi}')
        if mse < epsilon_squared:
            lo = mid
            bound_dict['lo'] = (mse, z, ghat)
        elif mse > epsilon_squared:
            hi = mid
            bound_dict['hi'] = (mse, z, ghat)
        else:
            return (mid, mse, z, ghat)
        num_evals -= 1
    return (hi,) + bound_dict['hi'] if is_pos else (lo,) + bound_dict['lo']

            

# def optimize_alpha(vanilla_dy_dx, zo_dy_dx, net, criterion, method, gt_data, 
#                    label_pred, num_attack_iterations, num_dummy, imidx_list,
#                    num_alpha_search_iterations, epsilon_squared):
#      inv_attack_closure = lambda ghat: inv_attack(ghat, net, criterion, method, gt_data, label_pred, 
#                                        num_attack_iterations, None, num_dummy, None, 
#                                        imidx_list, None, False)
def inv_attack_closure(alpha):
    return alpha * alpha, 'z', 'ghat'
def get_bisection_search_eval_fn(sign):
    if sign == 'pos':
        def bisection_search_eval_fn(alpha):
            return inv_attack_closure(alpha)
    elif sign == 'neg':
        def bisection_search_eval_fn(alpha):
            mse, z, ghat = inv_attack_closure(alpha)
            return -mse, z, ghat
    else:
        raise ValueError('sign must be either `pos` or `neg')
    return bisection_search_eval_fn

            
epsilon_squared = 0.6
num_alpha_search_iterations = 10
mse_0, z_0, ghat_0 = inv_attack_closure(0)
if mse_0 >= epsilon_squared:
    # if the vanilla gradient (alpha=0) is already larger than the error tol, then we are satisfying the constraint and can't reduce alpha any further. return 
    print( 0, ghat_0)
alpha_star_pos, mse_star_pos, zhat_pos, ghat_alpha_star_pos = bisection_search(0, 1, get_bisection_search_eval_fn('pos'), num_alpha_search_iterations, epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True)
alpha_star_neg, mse_star_neg, zhat_neg, ghat_alpha_star_neg = bisection_search(-1, 0, get_bisection_search_eval_fn('neg'), num_alpha_search_iterations, -epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True)


Initial bounds: (0, 1)
10/10 evaluations remaining after initialization.
alpha:0.5, mse:0.25, lo:0, hi:1
alpha:0.75, mse:0.5625, lo:0.5, hi:1
alpha:0.875, mse:0.765625, lo:0.75, hi:1
alpha:0.8125, mse:0.66015625, lo:0.75, hi:0.875
alpha:0.78125, mse:0.6103515625, lo:0.75, hi:0.8125
alpha:0.765625, mse:0.586181640625, lo:0.75, hi:0.78125
alpha:0.7734375, mse:0.59820556640625, lo:0.765625, hi:0.78125
alpha:0.77734375, mse:0.6042633056640625, lo:0.7734375, hi:0.78125
alpha:0.775390625, mse:0.6012306213378906, lo:0.7734375, hi:0.77734375
alpha:0.7744140625, mse:0.5997171401977539, lo:0.7734375, hi:0.775390625
Initial bounds: (-1, 0)
10/10 evaluations remaining after initialization.
alpha:-0.5, mse:-0.25, lo:-1, hi:0
alpha:-0.75, mse:-0.5625, lo:-1, hi:-0.5
alpha:-0.875, mse:-0.765625, lo:-1, hi:-0.75
alpha:-0.8125, mse:-0.66015625, lo:-0.875, hi:-0.75
alpha:-0.78125, mse:-0.6103515625, lo:-0.8125, hi:-0.75
alpha:-0.765625, mse:-0.586181640625, lo:-0.78125, hi:-0.75
alpha:-0.7734375, mse:-0

In [16]:
alpha_star_pos, mse_star_pos

(0.775390625, 0.6012306213378906)

In [17]:
alpha_star_neg, mse_star_neg

(-0.775390625, -0.6012306213378906)